In [0]:
# Load all Bronze tables as PySpark DataFrames
customers = spark.table("workspace.ecommerce.bronze_customers")
products = spark.table("workspace.ecommerce.bronze_products")
orders = spark.table("workspace.ecommerce.bronze_orders")
order_items = spark.table("workspace.ecommerce.bronze_order_items")

# Quick count of all tables
print(f"Customers: {customers.count()}")
print(f"Products:  {products.count()}")
print(f"Orders:    {orders.count()}")
print(f"Order Items: {order_items.count()}")

Customers: 200
Products:  50
Orders:    1000
Order Items: 2000


In [0]:
# Look at the schema of all 4 tables
print("=== CUSTOMERS SCHEMA ===")
customers.printSchema()

print("=== PRODUCTS SCHEMA ===")
products.printSchema()

print("=== ORDERS SCHEMA ===")
orders.printSchema()

print("=== ORDER ITEMS SCHEMA ===")
order_items.printSchema()

=== CUSTOMERS SCHEMA ===
root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: string (nullable = true)

=== PRODUCTS SCHEMA ===
root
 |-- product_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- base_price: string (nullable = true)

=== ORDERS SCHEMA ===
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)

=== ORDER ITEMS SCHEMA ===
root
 |-- item_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- total_price: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, count, when, isnan

# Check for nulls across all columns in orders
print("=== NULL CHECK — ORDERS ===")
orders.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in orders.columns
]).show()

print("=== NULL CHECK — ORDER ITEMS ===")
order_items.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in order_items.columns
]).show()

print("=== NULL CHECK — CUSTOMERS ===")
customers.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in customers.columns
]).show()

=== NULL CHECK — ORDERS ===
+--------+-----------+----------+------+
|order_id|customer_id|order_date|status|
+--------+-----------+----------+------+
|       0|          0|         0|     0|
+--------+-----------+----------+------+

=== NULL CHECK — ORDER ITEMS ===
+-------+--------+----------+--------+----------+-----------+
|item_id|order_id|product_id|quantity|unit_price|total_price|
+-------+--------+----------+--------+----------+-----------+
|      0|       0|         0|       0|         0|          0|
+-------+--------+----------+--------+----------+-----------+

=== NULL CHECK — CUSTOMERS ===
+-----------+----+-----+-------+-----------+
|customer_id|name|email|country|signup_date|
+-----------+----+-----+-------+-----------+
|          0|   0|    0|      0|          0|
+-----------+----+-----+-------+-----------+



In [0]:
from pyspark.sql.functions import lit
from pyspark.sql import Row
import random

# Simulate real-world data quality issues
# Introduce nulls into 5% of customer emails
customers_messy = customers.withColumn(
    "email",
    when(col("customer_id") % 20 == 0, lit(None)).otherwise(col("email"))
)

# Introduce nulls into some order dates
orders_messy = orders.withColumn(
    "order_date",
    when(col("order_id") % 25 == 0, lit(None)).otherwise(col("order_date"))
)

# Verify the nulls were introduced
print("=== MESSY CUSTOMERS — NULL CHECK ===")
customers_messy.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in customers_messy.columns
]).show()

print("=== MESSY ORDERS — NULL CHECK ===")
orders_messy.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in orders_messy.columns
]).show()

=== MESSY CUSTOMERS — NULL CHECK ===
+-----------+----+-----+-------+-----------+
|customer_id|name|email|country|signup_date|
+-----------+----+-----+-------+-----------+
|          0|   0|   10|      0|          0|
+-----------+----+-----+-------+-----------+

=== MESSY ORDERS — NULL CHECK ===
+--------+-----------+----------+------+
|order_id|customer_id|order_date|status|
+--------+-----------+----------+------+
|       0|          0|        40|     0|
+--------+-----------+----------+------+



In [0]:
# Calculate data quality scores
total_customers = customers_messy.count()
total_orders = orders_messy.count()

null_emails = customers_messy.filter(col("email").isNull()).count()
null_dates = orders_messy.filter(col("order_date").isNull()).count()

email_quality = ((total_customers - null_emails) / total_customers) * 100
date_quality = ((total_orders - null_dates) / total_orders) * 100

print("=" * 45)
print("       DATA QUALITY REPORT")
print("=" * 45)
print(f"\nCUSTOMERS TABLE ({total_customers} rows)")
print(f"  Email completeness:   {email_quality:.1f}%  {'✅' if email_quality > 95 else '⚠️'}")

print(f"\nORDERS TABLE ({total_orders} rows)")
print(f"  Date completeness:    {date_quality:.1f}%  {'✅' if date_quality > 95 else '⚠️'}")

print("\n" + "=" * 45)
print("RECOMMENDATION:")
if email_quality <= 95:
    print(f"  ⚠️  {null_emails} customers missing email — exclude from marketing campaigns")
if date_quality <= 95:
    print(f"  ⚠️  {null_dates} orders missing date — exclude from trend analysis")
print("=" * 45)

       DATA QUALITY REPORT

CUSTOMERS TABLE (200 rows)
  Email completeness:   95.0%  ⚠️

ORDERS TABLE (1000 rows)
  Date completeness:    96.0%  ✅

RECOMMENDATION:
  ⚠️  10 customers missing email — exclude from marketing campaigns


In [0]:
from pyspark.sql.functions import lower, trim, to_date

# Silver transformation -- clean the messy data
customers_clean = customers_messy.filter(
    col("email").isNotNull()  # drop rows with missing email
).withColumn(
    "email", lower(trim(col("email")))  # normalize email to lowercase
).withColumn(
    "signup_date", to_date(col("signup_date"))  # cast string to date
)

orders_clean = orders_messy.filter(
    col("order_date").isNotNull()  # drop rows with missing order date
).withColumn(
    "order_date", to_date(col("order_date"))  # cast string to date
)

# Verify the fix
print("=== AFTER CLEANING ===")
print(f"Customers: {customers_messy.count()} → {customers_clean.count()} rows")
print(f"Orders:    {orders_messy.count()} → {orders_clean.count()} rows")

print("\n=== CLEANED CUSTOMERS SCHEMA ===")
customers_clean.printSchema()

print("\n=== SAMPLE CLEAN DATA ===")
customers_clean.show(5)

=== AFTER CLEANING ===
Customers: 200 → 190 rows
Orders:    1000 → 960 rows

=== CLEANED CUSTOMERS SCHEMA ===
root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)


=== SAMPLE CLEAN DATA ===
+-----------+-----------------+--------------------+----------------+-----------+
|customer_id|             name|               email|         country|signup_date|
+-----------+-----------------+--------------------+----------------+-----------+
|          1|       Brian Yang|johnsonjoshua@exa...|         Mayotte| 2023-07-17|
|          2|    Lance Hoffman|fjohnson@example.org|Saint Barthelemy| 2025-03-25|
|          3|  Abigail Shaffer|lrobinson@example...|           Qatar| 2024-07-19|
|          4|       Ian Cooper|  eric51@example.org|         Estonia| 2025-09-13|
|          5|Sandra Montgomery|yherrera@example.org|         Tokelau| 2024-05-13|
+------

In [0]:
from pyspark.sql.functions import datediff, current_date, when, col

# Add derived columns to enrich the Silver layer
customers_enriched = customers_clean.withColumn(
    # How many days since customer signed up
    "days_since_signup", datediff(current_date(), col("signup_date"))
).withColumn(
    # Customer tenure category
    "tenure_segment",
    when(datediff(current_date(), col("signup_date")) < 90, "New")
    .when(datediff(current_date(), col("signup_date")) < 365, "Growing")
    .otherwise("Loyal")
)

# Show the enriched data
print("=== ENRICHED CUSTOMER DATA ===")
customers_enriched.select(
    "customer_id", "name", "signup_date", 
    "days_since_signup", "tenure_segment"
).show(10)

# How many customers in each segment?
print("=== CUSTOMER SEGMENTS ===")
customers_enriched.groupBy("tenure_segment") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

=== ENRICHED CUSTOMER DATA ===
+-----------+-----------------+-----------+-----------------+--------------+
|customer_id|             name|signup_date|days_since_signup|tenure_segment|
+-----------+-----------------+-----------+-----------------+--------------+
|          1|       Brian Yang| 2023-07-17|             1002|         Loyal|
|          2|    Lance Hoffman| 2025-03-25|              385|         Loyal|
|          3|  Abigail Shaffer| 2024-07-19|              634|         Loyal|
|          4|       Ian Cooper| 2025-09-13|              213|       Growing|
|          5|Sandra Montgomery| 2024-05-13|              701|         Loyal|
|          6|   Victoria Wyatt| 2023-08-01|              987|         Loyal|
|          7|    Michael Miles| 2025-03-23|              387|         Loyal|
|          8|   William Bowman| 2023-12-09|              857|         Loyal|
|          9|   Jennifer Jones| 2026-02-09|               64|           New|
|         10|     Derek Wright| 2024-11-23|  

In [0]:
# Write enriched customers as a new Silver table
customers_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ecommerce.silver_customers_enriched")

print("✅ silver_customers_enriched saved to Unity Catalog!")
print(f"   Rows: {customers_enriched.count()}")
print(f"   Columns: {len(customers_enriched.columns)}")
print(f"   New columns added: days_since_signup, tenure_segment")

✅ silver_customers_enriched saved to Unity Catalog!
   Rows: 190
   Columns: 7
   New columns added: days_since_signup, tenure_segment


In [0]:
from pyspark.sql.functions import sum, avg, round

# Load the full silver_order_items for spend data
silver = spark.table("workspace.ecommerce.silver_order_items")

# Join enriched customers with their spend data
customer_segments = customers_enriched.join(
    silver.groupBy("customer_id")
          .agg(
              sum("total_price").alias("total_spend"),
              count("*").alias("total_orders"),
              avg("total_price").alias("avg_order_value")
          ),
    on="customer_id",
    how="left"
).select(
    "customer_id", "name", "country",
    "tenure_segment", "days_since_signup",
    "total_spend", "total_orders", "avg_order_value"
)

# Save as Gold table
customer_segments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ecommerce.gold_customer_segments")

print("✅ gold_customer_segments saved!")
display(customer_segments.orderBy("total_spend", ascending=False).limit(10))

✅ gold_customer_segments saved!


customer_id,name,country,tenure_segment,days_since_signup,total_spend,total_orders,avg_order_value
130,Ashley Pena,Turkey,Growing,255,14494.02,23,630.1747826086956
150,Jeremy Turner,Falkland Islands (Malvinas),Loyal,449,14228.229999999998,17,836.9547058823528
196,Patricia Baker,Cote dIvoire,Growing,257,13152.81,34,386.84735294117644
55,Natasha Wells,Eritrea,Growing,274,12399.210000000001,21,590.4385714285714
168,Raven Taylor,France,Growing,92,11425.68,20,571.284
194,Denise Weber,Nicaragua,Loyal,513,11414.050000000001,17,671.414705882353
10,Derek Wright,Monaco,Loyal,507,11308.41,13,869.8776923076923
179,Cody Holt,Heard Island and McDonald Islands,New,84,10851.409999999998,20,542.5704999999999
21,John Ryan,Saint Kitts and Nevis,New,72,10703.990000000002,17,629.6464705882354
117,Kristina Herman,Cyprus,Loyal,388,10592.570000000002,21,504.4080952380953


In [0]:
from pyspark.sql.functions import sum, round

# Revenue contribution by tenure segment
segment_revenue = customer_segments.groupBy("tenure_segment") \
    .agg(
        round(sum("total_spend"), 2).alias("total_revenue"),
        count("*").alias("customer_count"),
        round(avg("avg_order_value"), 2).alias("avg_order_value")
    ).orderBy("total_revenue", ascending=False)

print("=== REVENUE BY CUSTOMER SEGMENT ===")
display(segment_revenue)

=== REVENUE BY CUSTOMER SEGMENT ===


tenure_segment,total_revenue,customer_count,avg_order_value
Loyal,580002.62,127,460.59
Growing,195418.18,41,503.97
New,116817.33,22,479.42


Databricks visualization. Run in Databricks to view.